Model Evaluation
In this notebook, we are going to work on one of the most practical topics in AI Engineering: Model Evaluation for Code Generation.

In many projects, people stop after the model generates code. But in real AI engineering, generation alone is not enough. We must verify whether the output is actually correct, executable, and efficient.

That is the purpose of this notebook.

Here, we will take a benchmark-style Python function, ask different models to convert it into Rust, and then evaluate the generated result using measurable signals such as:

    generation time
    compilation success
    execution success
    output correctness
    runtime of Python code
    runtime of generated Rust code
    Rust speedup over Python

This means our evaluation will not be based only on how good the output looks. Instead, it will be based on whether the generated code works correctly and whether it provides performance benefits.

We will also compare multiple model families in the same workflow, including:

    Hugging Face open models
    GPT models
    Claude models
    Gemini models

By the end of this notebook, you will understand how to evaluate code-generation models using a structured and engineering-focused pipeline.

Section 1 — Environment setup

We begin by installing the required packages for:

    Hugging Face local inference
    provider SDKs
    Rust compilation
    Gradio interface


In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece gradio pandas openai anthropic google-genai
!apt-get -qq update
!apt-get -qq install -y rustc cargo

print("Environment setup completed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 11.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libstd-rust-1.75:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libstd-rust-1.75_1.75.0+dfsg0ubuntu1~bpo0-0ubuntu0.22.04_amd64.deb ...
Unpacking libstd-rust-1.75:amd64 (1.75.0+dfsg0ubuntu1~bpo0-0ubuntu0.22.04) ...
Selecting previously unselected package libstd-rust-dev:amd64.
Preparing to unpack .../libstd-rust-dev_1.75.0+dfsg0ubuntu1~bpo0-0ubuntu0.22.04_amd64.deb ...
Unpacking libstd-rust-dev:amd64 (1.75.0+dfsg0ubuntu1~bpo0-0ubuntu0.22.04) ...
Selecting previously unselected package rustc.
Preparing to unpack .../rustc_1.75.0+dfsg0ubuntu1~bpo0-0ub

Section 2 — Runtime verification

Before loading any model or running any benchmark, we should confirm that the runtime has the tools we need.

In this notebook, two checks are especially important:

    GPU availability
    This matters when we want to run a local Hugging Face model efficiently. Without a GPU, local inference may become too slow or may not fit properly.

    Rust availability
    This matters because our evaluation is not only about generation. We also want to compile and run the generated Rust code. So Rust must be installed and working.

This section helps us verify that the runtime is ready for both:

    model inference
    compilation-based evaluation


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


Section 3 — Import required modules

These imports support:

    model inference
    code extraction
    compilation
    evaluation
    Gradio UI


In [4]:
import os
import re
import gc
import json
import time
import subprocess
import pandas as pd
import gradio as gr

from google.colab import userdata

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from openai import OpenAI
import anthropic
from google import genai

Section 4 — Load API keys from Colab Secrets

This section is used only when you want to evaluate provider-hosted models such as:

    OpenAI
    Anthropic
    Gemini

To use those providers, you need API keys. In Google Colab, the safest way is to store them in Colab Secrets instead of writing them directly in the notebook.

Create these secrets in Colab if you want to use hosted provider APIs:

    OPENAI_API_KEY
    ANTHROPIC_API_KEY
    GEMINI_API_KEY

If you only want to test a local Hugging Face model, then these secrets are not required and can stay empty.

So this section is basically the secure bridge between your notebook and external model providers.

In [5]:
def get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
GEMINI_API_KEY = get_secret("GEMINI_API_KEY")

print("OPENAI key loaded:", OPENAI_API_KEY is not None)
print("ANTHROPIC key loaded:", ANTHROPIC_API_KEY is not None)
print("GEMINI key loaded:", GEMINI_API_KEY is not None)

OPENAI key loaded: True
ANTHROPIC key loaded: True
GEMINI key loaded: True


Section 5 — Provider clients

In this section, we create client objects for each hosted model provider.

A client is the Python object that allows our notebook to send requests to that provider’s model API.

We create a client only if its corresponding API key is available. That keeps the notebook flexible:

    if a key is present, that provider can be used
    if a key is missing, that provider is simply skipped

This lets us support multiple providers in one notebook without forcing all of them to be active at the same time.

Technically, each provider has its own SDK style:

    OpenAI uses its Python SDK
    Anthropic uses its Messages API through its SDK
    Gemini uses Google’s GenAI SDK

This section sets up those connections so later the notebook can call all providers in a unified evaluation flow.

In [6]:
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

Model Shortlisting Strategy

Before running the practical benchmark, we usually do not test every possible model in the world. That would be expensive, slow, and difficult to manage.

So first, we shortlist a few promising models based on external benchmark references such as Artificial Analysis.

This helps us choose models that already look strong in areas like:

    intelligence
    speed
    price
    coding-related quality

But external rankings are still only a starting point.

A model may look strong on a leaderboard and still perform poorly on your specific engineering task. That is why this notebook performs task-level evaluation.

Here, we are checking whether the shortlisted models can actually generate Rust code that:

    compiles successfully
    runs without failure
    produces the correct output
    improves runtime compared to the original Python version

So in this workflow:

    external benchmarks help us shortlist
    this notebook helps us validate in practice

That is a very important distinction in real AI engineering.

Section 6 — Model registry

In this section, we define all model options in one central place called the model registry.

A model registry is simply a dictionary that stores useful metadata about each model, such as:

    a readable label
    the provider type
    the actual model ID

This makes the notebook cleaner and easier to maintain.

Instead of hardcoding model names again and again in different cells, we define them once here and reuse them everywhere later in the notebook.

This is useful because:

    it reduces repetition
    it makes switching models easier
    it keeps the evaluation loop consistent across providers

Important usage note:

    Use a Hugging Face model if you want local notebook-side inference
    Use OpenAI, Claude, or Gemini if you want hosted API inference
    The rest of the evaluation flow stays the same

So this section acts like the configuration center for all models used in the notebook.

In [7]:
MODEL_REGISTRY = {
    "hf_qwen_coder": {
        "label": "Hugging Face - Qwen Coder 1.5B",
        "type": "huggingface",
        "model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    },
    "openai_gpt": {
        "label": "OpenAI - GPT",
        "type": "openai",
        "model_id": "gpt-5",
    },
    "google_gemini_flash": {
        "label": "Google - Gemini 2.5 Flash",
        "type": "gemini",
        "model_id": "gemini-2.5-flash",
    },
    "anthropic_sonnet_46": {
        "label": "Anthropic - Claude Sonnet 4.6",
        "type": "anthropic",
        "model_id": "claude-sonnet-4-6",
    },
}

print(json.dumps(MODEL_REGISTRY, indent=2))

{
  "hf_qwen_coder": {
    "label": "Hugging Face - Qwen Coder 1.5B",
    "type": "huggingface",
    "model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct"
  },
  "openai_gpt": {
    "label": "OpenAI - GPT",
    "type": "openai",
    "model_id": "gpt-5"
  },
  "google_gemini_flash": {
    "label": "Google - Gemini 2.5 Flash",
    "type": "gemini",
    "model_id": "gemini-2.5-flash"
  },
  "anthropic_sonnet_46": {
    "label": "Anthropic - Claude Sonnet 4.6",
    "type": "anthropic",
    "model_id": "claude-sonnet-4-6"
  }
}


Section 7 — Hugging Face quantization config

When we load a local Hugging Face model in Colab, memory usage becomes very important.

Large language models can consume a lot of GPU memory. To reduce that memory pressure, we use quantization.

In simple terms, quantization means storing model weights in a more compact way so the model uses less memory.

Here, we use a 4-bit configuration with BitsAndBytes. This helps us:

    load the model more efficiently
    fit larger models into limited Colab GPU memory
    still run local inference for benchmarking

So this section prepares an optimized loading setup for local Hugging Face models.

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("BitsAndBytes config ready.")

BitsAndBytes config ready.


Section 8 — Hugging Face model loader

This section handles loading local/open models from Hugging Face.

Why do we need a dedicated loader?

Because loading a model repeatedly is expensive. It takes time and memory. Also, if we keep multiple models in memory, Colab may run out of GPU RAM.

So this section does two useful things:

    it unloads any previously loaded model to free memory
    it loads the requested Hugging Face model only when needed

This keeps the notebook more stable and helps reduce memory pressure during benchmarking.

Technically, this section manages:

    tokenizer loading
    model loading
    current model tracking
    cleanup of old model objects

So this is the memory-management part of the local inference workflow.

In [9]:
current_model = None
current_tokenizer = None
current_model_name = None

def unload_model():
    global current_model, current_tokenizer, current_model_name
    current_model = None
    current_tokenizer = None
    current_model_name = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_hf_model(model_key):
    global current_model, current_tokenizer, current_model_name

    cfg = MODEL_REGISTRY[model_key]
    model_id = cfg["model_id"]

    if current_model_name == model_key and current_model is not None:
        return current_tokenizer, current_model

    unload_model()

    print(f"Loading Hugging Face model: {cfg['label']}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )

    current_tokenizer = tokenizer
    current_model = model
    current_model_name = model_key

    return tokenizer, model

Section 9 — Shared prompt template

In this section, we define one common prompt template that will be used for all models.

This is very important for fair evaluation.

If we give different prompts to different models, then the benchmark becomes less reliable, because changes in output may come from the prompt rather than the model itself.

So we use one shared prompt template to keep the task consistent across providers.

The prompt clearly tells the model:

    convert the given Python function into Rust
    use the exact Rust function signature
    return only code
    do not add explanations
    do not add Markdown fences
    do not add a main function

This makes the generated output easier to compare and easier to compile later.

So this section is about fairness and consistency in prompting.

In [10]:
def build_prompt(python_code, rust_signature, extra_instruction=""):
    return f"""
You are an expert systems programmer.

Convert the following Python function into Rust.

Requirements:
1. Return only valid Rust code
2. Use the exact Rust function signature provided below
3. Do not include Markdown code fences
4. Do not include commentary or explanation
5. Do not include a main function
6. Keep the implementation correct and idiomatic
7. Preserve the same algorithmic approach unless explicitly told otherwise

Rust function signature:
{rust_signature}

Additional instruction:
{extra_instruction}

Python function:
{python_code}
""".strip()

Section 10 — Output cleanup

Some models return fenced blocks or extra text. This helper normalizes the response into pure Rust code.

It will clean something like:

    Markdown code blocks
    extra text
    explanations


In [11]:
def extract_code(text):
    blocks = re.findall(r"```(?:rust)?\s*(.*?)```", text, flags=re.DOTALL)
    if blocks:
        return blocks[0].strip()
    return text.strip()

Section 11 — Hugging Face generation function

This function is used when a Hugging Face model is selected.

In [12]:
def generate_with_huggingface(model_key, python_code, rust_signature, extra_instruction=""):
    tokenizer, model = load_hf_model(model_key)
    prompt = build_prompt(python_code, rust_signature, extra_instruction)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return extract_code(text)

Section 12 — OpenAI generation function

This function uses OpenAI’s Python SDK and Responses API pattern.

In [13]:
def generate_with_openai(model_key, python_code, rust_signature, extra_instruction=""):
    if openai_client is None:
        raise ValueError("OPENAI_API_KEY is not available in Colab Secrets.")

    model_id = MODEL_REGISTRY[model_key]["model_id"]
    prompt = build_prompt(python_code, rust_signature, extra_instruction)

    response = openai_client.responses.create(
        model=model_id,
        input=prompt
    )

    text = response.output_text
    return extract_code(text)

Section 13 — Anthropic generation function

This function uses Anthropic’s Messages API structure.

In [14]:
def generate_with_anthropic(model_key, python_code, rust_signature, extra_instruction=""):
    if anthropic_client is None:
        raise ValueError("ANTHROPIC_API_KEY is not available in Colab Secrets.")

    model_id = MODEL_REGISTRY[model_key]["model_id"]
    prompt = build_prompt(python_code, rust_signature, extra_instruction)

    response = anthropic_client.messages.create(
        model=model_id,
        max_tokens=1000,
        temperature=0.2,
        system="You are an expert systems programmer.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    text = ""
    for block in response.content:
        if getattr(block, "type", None) == "text":
            text += block.text

    return extract_code(text)

Section 14 — Gemini generation function

This function uses Google’s official GenAI SDK pattern with generate_content.

In [15]:
def generate_with_gemini(model_key, python_code, rust_signature, extra_instruction=""):
    if gemini_client is None:
        raise ValueError("GEMINI_API_KEY is not available in Colab Secrets.")

    model_id = MODEL_REGISTRY[model_key]["model_id"]
    prompt = build_prompt(python_code, rust_signature, extra_instruction)

    response = gemini_client.models.generate_content(
        model=model_id,
        contents=prompt
    )

    text = response.text
    return extract_code(text)

Section 15 — Unified generation router

Up to this point, we have separate generation functions for:

    Hugging Face
    OpenAI
    Anthropic
    Gemini

Now we need one clean way to call them.

That is the purpose of the unified generation router.

This router checks the selected model’s provider type and then automatically sends the request to the correct generation function.

It also records the total generation time.

This is useful because the rest of the notebook does not need to know provider-specific details anymore. It can simply say:

    “generate Rust for this model”

And the router handles the rest.

So this section is what makes the full pipeline cleaner, simpler, and easier to scale.

In [16]:
def generate_rust(model_key, python_code, rust_signature, extra_instruction=""):
    provider_type = MODEL_REGISTRY[model_key]["type"]

    start_time = time.time()

    if provider_type == "huggingface":
        rust_code = generate_with_huggingface(model_key, python_code, rust_signature, extra_instruction)
    elif provider_type == "openai":
        rust_code = generate_with_openai(model_key, python_code, rust_signature, extra_instruction)
    elif provider_type == "anthropic":
        rust_code = generate_with_anthropic(model_key, python_code, rust_signature, extra_instruction)
    elif provider_type == "gemini":
        rust_code = generate_with_gemini(model_key, python_code, rust_signature, extra_instruction)
    else:
        raise ValueError(f"Unsupported provider type: {provider_type}")

    generation_time = time.time() - start_time
    return rust_code, generation_time

Section 16 — Rust compilation, execution, and runtime benchmarking

This is one of the most important sections in the notebook.

Why? Because real model evaluation should not stop at code generation.

After a model generates Rust code, we want to know:

    does it compile?
    does it run?
    does it produce the correct output?
    how fast is it?

So this section takes the generated Rust function, wraps it with a small main function, compiles it using rustc, runs the executable, and records the result.

It captures important benchmark signals such as:

    compile success
    run success
    final Rust output
    Rust execution time
    compiler/runtime errors if something fails

This is the section that turns code generation into a true engineering benchmark.

In [17]:
def compile_and_run_rust_benchmark(rust_function_code, task):
    benchmark_limit = task["benchmark_limit"]

    rust_fn_name = task["rust_signature"].split("(")[0].replace("fn", "").strip()

    rust_main = f"""
use std::time::Instant;

{rust_function_code}

fn main() {{
    let limit: usize = {benchmark_limit};

    let start = Instant::now();
    let result = {rust_fn_name}(limit);
    let elapsed = start.elapsed().as_secs_f64();

    println!("RESULT={{}}", result);
    println!("SECONDS={{:.6}}", elapsed);
}}
""".strip()

    full_code = rust_main

    with open("main.rs", "w", encoding="utf-8") as f:
        f.write(full_code)

    compile_start = time.time()
    compile_result = subprocess.run(
        ["rustc", "main.rs", "-O", "-o", "main"],
        capture_output=True,
        text=True
    )
    compile_end = time.time()

    if compile_result.returncode != 0:
        return {
            "compile_success": False,
            "run_success": False,
            "rust_result": None,
            "rust_execution_time_seconds": None,
            "compile_time_seconds": round(compile_end - compile_start, 4),
            "stderr": compile_result.stderr,
            "full_code": full_code
        }

    run_result = subprocess.run(
        ["./main"],
        capture_output=True,
        text=True
    )

    if run_result.returncode != 0:
        return {
            "compile_success": True,
            "run_success": False,
            "rust_result": None,
            "rust_execution_time_seconds": None,
            "compile_time_seconds": round(compile_end - compile_start, 4),
            "stderr": run_result.stderr,
            "full_code": full_code
        }

    rust_result = None
    rust_execution_time_seconds = None

    for line in run_result.stdout.strip().splitlines():
        if line.startswith("RESULT="):
            rust_result = int(line.split("=", 1)[1].strip())
        elif line.startswith("SECONDS="):
            rust_execution_time_seconds = float(line.split("=", 1)[1].strip())

    return {
        "compile_success": True,
        "run_success": True,
        "rust_result": rust_result,
        "rust_execution_time_seconds": rust_execution_time_seconds,
        "compile_time_seconds": round(compile_end - compile_start, 4),
        "stderr": "",
        "full_code": full_code
    }

Section 17 — Python baseline benchmark

Before we compare model-generated Rust code, we first need to understand the original Python version.

This is called the baseline.

A baseline gives us the reference point for the entire evaluation. Without it, we would not know:

    what the correct output should be
    how long the original Python code takes
    whether the Rust output is actually faster

So in this step, we run the benchmark-style Python function directly and observe its result and runtime.

This is very important because later all generated Rust versions will be compared against this same original implementation.

In practical AI engineering, a good baseline is essential.
You cannot measure improvement unless you first know the starting point.

In [18]:
import time

slow_python_code = """
def count_primes_python(limit):
    count = 0
    for num in range(2, limit + 1):
        is_prime = True
        for i in range(2, int(num ** 0.5) + 1):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            count += 1
    return count
"""

exec(slow_python_code)

print("Running Python Baseline... (This might take 3-5 seconds)")
start_time = time.time()
result = count_primes_python(1000000)
end_time = time.time()

print(f"Result: {result} primes found.")
print(f"Python Execution Time: {end_time - start_time:.4f} seconds")

Running Python Baseline... (This might take 3-5 seconds)
Result: 78498 primes found.
Python Execution Time: 4.4779 seconds


Section 18 — Python baseline

This section turns the Python baseline idea into a reusable function.

Instead of running the Python code manually every time, we wrap it into a helper function that:

    loads the Python function
    runs it on the benchmark input
    records the output
    records the execution time

This is useful because later we will evaluate multiple models, and for each evaluation we want a consistent Python reference.

So this section creates the reusable baseline step that the full benchmark loop depends on.

In [19]:
def run_python_baseline(task):
    local_scope = {}
    exec(task["python_code"], {}, local_scope)

    python_function = local_scope[task["python_function_name"]]
    limit = task["benchmark_limit"]

    print("Running Python baseline...")
    start_time = time.time()
    result = python_function(limit)
    end_time = time.time()

    return {
        "python_result": result,
        "python_execution_time_seconds": round(end_time - start_time, 4)
    }

Section 19 — Evaluate one model

This section combines the previous building blocks and evaluates one selected model from start to finish.

For one model, the notebook performs the full workflow:

    run the Python baseline
    ask the model to generate Rust code
    compile the generated Rust
    run the compiled program
    compare the Rust result with the Python result
    calculate performance metrics

It records signals such as:

    generation time
    compile success
    run success
    output correctness
    Python runtime
    Rust runtime
    Rust speedup over Python

So this section is the first complete end-to-end evaluation unit in the notebook.

In [20]:
BENCHMARK_TASK = {
    "task_name": "count_primes",
    "python_function_name": "count_primes_python",
    "python_code": slow_python_code,
    "rust_signature": "fn count_primes_python(limit: usize) -> usize",
    "extra_instruction": "Return only the Rust function. No markdown fences. No explanation.",
    "benchmark_limit": 1000000,
    "expected_output": 41538
}

def evaluate_model(model_key, task):
    python_baseline = run_python_baseline(task)

    rust_code, generation_time = generate_rust(
        model_key=model_key,
        python_code=task["python_code"],
        rust_signature=task["rust_signature"],
        extra_instruction=task["extra_instruction"]
    )

    rust_result = compile_and_run_rust_benchmark(rust_code, task)

    output_match = (
        rust_result["compile_success"]
        and rust_result["run_success"]
        and rust_result["rust_result"] == python_baseline["python_result"]
    )

    python_time = python_baseline["python_execution_time_seconds"]
    rust_time = rust_result["rust_execution_time_seconds"]

    if rust_time is not None and rust_time > 0:
        speedup_vs_python = round(python_time / rust_time, 4)
    else:
        speedup_vs_python = None

    return pd.DataFrame([{
        "model_key": model_key,
        "model_label": MODEL_REGISTRY[model_key]["label"],
        "provider_type": MODEL_REGISTRY[model_key]["type"],
        "task": task["task_name"],
        "benchmark_limit": task["benchmark_limit"],
        "python_result": python_baseline["python_result"],
        "rust_result": rust_result["rust_result"],
        "compile_success": rust_result["compile_success"],
        "run_success": rust_result["run_success"],
        "output_match": output_match,
        "generation_time_seconds": round(generation_time, 4),
        "compile_time_seconds": rust_result["compile_time_seconds"],
        "python_execution_time_seconds": python_time,
        "rust_execution_time_seconds": rust_time,
        "speedup_vs_python": speedup_vs_python,
        "generated_rust": rust_code,
        "stderr": rust_result["stderr"][:1000]
    }])

["hf_qwen_coder", "openai_gpt", "google_gemini_flash", "anthropic_sonnet_46"]

['hf_qwen_coder', 'openai_gpt', 'google_gemini_flash', 'anthropic_sonnet_46']

Section 20 — Choose which models to evaluate

This section defines which models will participate in the benchmark.

All selected models will receive the same Python code and will be evaluated using the same compilation, execution, and runtime comparison process.

In [21]:
SELECTED_MODELS = [
    "hf_qwen_coder",
    "google_gemini_flash",
    "anthropic_sonnet_46",
    "openai_gpt",
]

SELECTED_MODELS

['hf_qwen_coder', 'google_gemini_flash', 'anthropic_sonnet_46', 'openai_gpt']

Section 21 — Run the evaluation across all selected models

Now that we have chosen the participating models, this section runs the complete benchmark for all of them.

This is the full multi-model evaluation loop.

For each selected model, the notebook:

    sends the same Python benchmark code
    asks the model to convert it into Rust
    compiles the generated Rust code
    runs the compiled program
    checks whether the output is correct
    records timing and evaluation details

All results are collected into one combined dataframe called results_df.

This is very important because fair comparison is only possible when all models are tested under the same task and the same evaluation logic.

So this section is where the full benchmark actually happens.

In [22]:
all_results = []

for model_key in SELECTED_MODELS:
    print("=" * 100)
    print(f"Evaluating: {MODEL_REGISTRY[model_key]['label']}")

    try:
        result_df = evaluate_model(model_key, BENCHMARK_TASK)

        model_label = MODEL_REGISTRY[model_key]["label"]
        generated_rust_code = result_df.iloc[0]["generated_rust"]

        print(f"\n{model_label} generated Rust code:\n")
        print(generated_rust_code)
        print("\n" + "=" * 100 + "\n")

    except Exception as e:
        result_df = pd.DataFrame([{
            "model_key": model_key,
            "model_label": MODEL_REGISTRY[model_key]["label"],
            "provider_type": MODEL_REGISTRY[model_key]["type"],
            "task": BENCHMARK_TASK["task_name"],
            "benchmark_limit": BENCHMARK_TASK["benchmark_limit"],
            "python_result": None,
            "rust_result": None,
            "compile_success": False,
            "run_success": False,
            "output_match": False,
            "generation_time_seconds": None,
            "compile_time_seconds": None,
            "python_execution_time_seconds": None,
            "rust_execution_time_seconds": None,
            "speedup_vs_python": None,
            "generated_rust": "",
            "stderr": str(e)
        }])

        print(f"\n{MODEL_REGISTRY[model_key]['label']} generated Rust code:\n")
        print("No code generated due to error.")
        print("Error:", e)
        print("\n" + "=" * 100 + "\n")

    all_results.append(result_df)

results_df = pd.concat(all_results, ignore_index=True)
results_df

Evaluating: Hugging Face - Qwen Coder 1.5B
Running Python baseline...
Loading Hugging Face model: Hugging Face - Qwen Coder 1.5B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Hugging Face - Qwen Coder 1.5B generated Rust code:

fn count_primes_python(limit: usize) -> usize {
    let mut count = 0;
    for num in 2..=limit {
        if is_prime(num) {
            count += 1;
        }
    }
    count
}

fn is_prime(n: usize) -> bool {
    if n <= 1 {
        return false;
    }
    for i in 2..=((n as f64).sqrt() as usize) {
        if n % i == 0 {
            return false;
        }
    }
    true
}


Evaluating: Google - Gemini 2.5 Flash
Running Python baseline...

Google - Gemini 2.5 Flash generated Rust code:

fn count_primes_python(limit: usize) -> usize {
    let mut count = 0;
    for num in 2..=limit {
        let mut is_prime = true;
        // The square root of num
        let sqrt_num = (num as f64).sqrt() as usize;
        for i in 2..=sqrt_num {
            if num % i == 0 {
                is_prime = false;
                break;
            }
        }
        if is_prime {
            count += 1;
        }
    }
    count
}


Evaluating: An

/tmp/ipykernel_2047/694257830.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(all_results, ignore_index=True)


,model_key,model_label,provider_type,task,benchmark_limit,python_result,rust_result,compile_success,run_success,output_match,generation_time_seconds,compile_time_seconds,python_execution_time_seconds,rust_execution_time_seconds,speedup_vs_python,generated_rust,stderr
0,hf_qwen_coder,Hugging Face - Qwen Coder 1.5B,huggingface,count_primes,1000000,78498,78498,True,True,True,78.3186,1.8458,8.8909,0.271177,32.7863,fn count_primes_python(limit: usize) -> usize ...,
1,google_gemini_flash,Google - Gemini 2.5 Flash,gemini,count_primes,1000000,78498,78498,True,True,True,3.1630,1.2432,5.7328,0.931014,6.1576,fn count_primes_python(limit: usize) -> usize ...,
2,anthropic_sonnet_46,Anthropic - Claude Sonnet 4.6,anthropic,count_primes,1000000,None,None,False,False,False,NaN,NaN,NaN,NaN,NaN,,"Error code: 400 - {'type': 'error', 'error': {..."
3,openai_gpt,OpenAI - GPT,openai,count_primes,1000000,78498,78498,True,True,True,13.1250,0.2732,4.4185,0.274618,16.0896,fn count_primes_python(limit: usize) -> usize ...,


Section 22 — Final comparison table

This table compares all selected models using:

    generation time
    compile success
    run success
    correctness
    Python runtime
    Rust runtime
    speedup over Python


In [23]:
summary_df = results_df[[
    "model_label",
    "provider_type",
    "benchmark_limit",
    "compile_success",
    "run_success",
    "output_match",
    "generation_time_seconds",
    "compile_time_seconds",
    "python_execution_time_seconds",
    "rust_execution_time_seconds",
    "speedup_vs_python"
]].copy()

summary_df

,model_label,provider_type,benchmark_limit,compile_success,run_success,output_match,generation_time_seconds,compile_time_seconds,python_execution_time_seconds,rust_execution_time_seconds,speedup_vs_python
0,Hugging Face - Qwen Coder 1.5B,huggingface,1000000,True,True,True,78.3186,1.8458,8.8909,0.271177,32.7863
1,Google - Gemini 2.5 Flash,gemini,1000000,True,True,True,3.1630,1.2432,5.7328,0.931014,6.1576
2,Anthropic - Claude Sonnet 4.6,anthropic,1000000,False,False,False,NaN,NaN,NaN,NaN,NaN
3,OpenAI - GPT,openai,1000000,True,True,True,13.1250,0.2732,4.4185,0.274618,16.0896


Section 22A — Human-readable interpretation

Tables are useful, but sometimes it is easier to understand benchmark results in a more readable text format.

That is what this section provides.

Instead of only showing dataframe rows, we print each model’s results one by one in a human-friendly way.

This makes it easier to quickly read:

    whether the model succeeded
    whether the output was correct
    how long generation took
    how fast Rust was compared to Python

So this section improves readability and helps with quick manual inspection of the results.

In [24]:
for _, row in summary_df.iterrows():
    print(f"Model: {row['model_label']}")
    print(f"Provider: {row['provider_type']}")
    print(f"Benchmark limit: {row['benchmark_limit']}")
    print(f"Compile success: {row['compile_success']}")
    print(f"Run success: {row['run_success']}")
    print(f"Correct output: {row['output_match']}")
    print(f"Generation time: {row['generation_time_seconds']:.4f} sec" if pd.notnull(row["generation_time_seconds"]) else "Generation time: N/A")
    print(f"Compile time: {row['compile_time_seconds']:.4f} sec" if pd.notnull(row["compile_time_seconds"]) else "Compile time: N/A")
    print(f"Python execution time: {row['python_execution_time_seconds']:.4f} sec" if pd.notnull(row["python_execution_time_seconds"]) else "Python execution time: N/A")

    if pd.notnull(row["rust_execution_time_seconds"]):
        print(f"Rust execution time: {row['rust_execution_time_seconds']:.6f} sec")
    else:
        print("Rust execution time: Not available")

    if pd.notnull(row["speedup_vs_python"]):
        print(f"Rust speedup/Python: {row['speedup_vs_python']:.4f}x")
    else:
        print("Rust speedup/Python: Not available")

    print("-" * 70)

Model: Hugging Face - Qwen Coder 1.5B
Provider: huggingface
Benchmark limit: 1000000
Compile success: True
Run success: True
Correct output: True
Generation time: 78.3186 sec
Compile time: 1.8458 sec
Python execution time: 8.8909 sec
Rust execution time: 0.271177 sec
Rust speedup/Python: 32.7863x
----------------------------------------------------------------------
Model: Google - Gemini 2.5 Flash
Provider: gemini
Benchmark limit: 1000000
Compile success: True
Run success: True
Correct output: True
Generation time: 3.1630 sec
Compile time: 1.2432 sec
Python execution time: 5.7328 sec
Rust execution time: 0.931014 sec
Rust speedup/Python: 6.1576x
----------------------------------------------------------------------
Model: Anthropic - Claude Sonnet 4.6
Provider: anthropic
Benchmark limit: 1000000
Compile success: False
Run success: False
Correct output: False
Generation time: N/A
Compile time: N/A
Python execution time: N/A
Rust execution time: Not available
Rust speedup/Python: Not av

Section 23 — Sonnet 4.6 meta-evaluation

Here, Claude Sonnet 4.6 acts as a judge model.

Instead of generating Rust code for the task, it reviews the benchmark results of all models and gives a ranking.

It considers signals such as:

    correctness
    compile success
    run success
    runtime
    speedup over Python
    Rust code quality

This can be useful when you want an AI-generated interpretation of the benchmark.

However, the most important thing to remember is this:

The real benchmark is still based on objective engineering signals like compile success, run success, correctness, and runtime.
The judge model is only an additional interpretation layer.

So this section adds a meta-evaluation step on top of the raw benchmark results.

In [25]:
def strip_fenced_json(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def judge_with_sonnet_46(results_df):
    if anthropic_client is None:
        raise ValueError("ANTHROPIC_API_KEY is not available in Colab Secrets.")

    judge_model_id = MODEL_REGISTRY["anthropic_sonnet_46"]["model_id"]

    compact_metrics = results_df[[
        "model_label",
        "compile_success",
        "run_success",
        "output_match",
        "generation_time_seconds",
        "python_execution_time_seconds",
        "rust_execution_time_seconds",
        "speedup_vs_python"
    ]].to_dict(orient="records")

    compact_code = results_df[[
        "model_label",
        "generated_rust",
        "stderr"
    ]].to_dict(orient="records")

    prompt = f"""
You are judging a code-generation benchmark.

The original Python benchmark task was converted into Rust by multiple models.
Your job is to decide which model performed best overall.

Priority order:
1. Correctness
2. Compile and run success
3. Rust runtime
4. Speedup over Python
5. Rust code quality and idiomaticity

Return JSON only in this format:
{{
  "winner_model_label": "...",
  "ranking": [
    {{"model_label": "...", "rank": 1, "why": "..."}}
  ],
  "overall_reasoning": "..."
}}

Metrics:
{json.dumps(compact_metrics, indent=2)}

Generated Rust code and errors:
{json.dumps(compact_code, indent=2)}
""".strip()

    response = anthropic_client.messages.create(
        model=judge_model_id,
        max_tokens=1000,
        temperature=0.1,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    text = ""
    for block in response.content:
        if getattr(block, "type", None) == "text":
            text += block.text

    cleaned = strip_fenced_json(text)
    return json.loads(cleaned)

judge_result = judge_with_sonnet_46(results_df)

print("\n" + "=" * 100)
print("SONNET 4.6 META-EVALUATION RESULT")
print("=" * 100)

print(f"\nWinner Model: {judge_result['winner_model_label']}")

print("\nModel Ranking:")
for item in judge_result["ranking"]:
    print("-" * 100)
    print(f"Rank       : {item['rank']}")
    print(f"Model      : {item['model_label']}")
    print(f"Reason     : {item['why']}")

print("\n" + "-" * 100)
print("Overall Reasoning:")
print(judge_result["overall_reasoning"])
print("=" * 100)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CZQhRuZpKdc9RmPNG8pQ6'}

In [33]:
def strip_fenced_json(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def judge_with_sonnet_46(results_df):
    if gemini_client is None:
        raise ValueError("GEMINI_API_KEY is not available in Colab Secrets.")

    judge_model_id = MODEL_REGISTRY["google_gemini_flash"]["model_id"]

    compact_metrics = results_df[[
        "model_label",
        "compile_success",
        "run_success",
        "output_match",
        "generation_time_seconds",
        "python_execution_time_seconds",
        "rust_execution_time_seconds",
        "speedup_vs_python"
    ]].to_dict(orient="records")

    compact_code = results_df[[
        "model_label",
        "generated_rust",
        "stderr"
    ]].to_dict(orient="records")

    prompt = f"""
You are judging a code-generation benchmark.

The original Python benchmark task was converted into Rust by multiple models.
Your job is to decide which model performed best overall.

Priority order:
1. Correctness
2. Compile and run success
3. Rust runtime
4. Speedup over Python
5. Rust code quality and idiomaticity

Return JSON only in this format:
{{
  "winner_model_label": "...",
  "ranking": [
    {{"model_label": "...", "rank": 1, "why": "..."}}
  ],
  "overall_reasoning": "..."
}}

Metrics:
{json.dumps(compact_metrics, indent=2)}

Generated Rust code and errors:
{json.dumps(compact_code, indent=2)}
""".strip()

    response = gemini_client.models.generate_content(
        model=judge_model_id,
        contents=prompt,
    )

    text = response.text
    cleaned = strip_fenced_json(text)
    return json.loads(cleaned)

judge_result = judge_with_sonnet_46(results_df)

print("\n" + "=" * 100)
print("Gemini 2.5 Flash META-EVALUATION RESULT")
print("=" * 100)

print(f"\nWinner Model: {judge_result['winner_model_label']}")

print("\nModel Ranking:")
for item in judge_result["ranking"]:
    print("-" * 100)
    print(f"Rank       : {item['rank']}")
    print(f"Model      : {item['model_label']}")
    print(f"Reason     : {item['why']}")

print("\n" + "-" * 100)
print("Overall Reasoning:")
print(judge_result["overall_reasoning"])
print("=" * 100)



Gemini 2.5 Flash META-EVALUATION RESULT

Winner Model: Hugging Face - Qwen Coder 1.5B

Model Ranking:
----------------------------------------------------------------------------------------------------
Rank       : 1
Model      : Hugging Face - Qwen Coder 1.5B
Reason     : This model delivered correct, compiling, and runnable Rust code that achieved the fastest execution time (0.271177 seconds) and the highest speedup over Python (32.7863x). Its code quality is also slightly superior due to the clear separation of the `is_prime` logic into its own helper function, making it more modular and readable.
----------------------------------------------------------------------------------------------------
Rank       : 2
Model      : OpenAI - GPT
Reason     : The model produced correct, compiling, and runnable Rust code. It achieved very good performance, with a Rust execution time of 0.274618 seconds, which is extremely close to the top performer, and a substantial speedup of 16.0896x. The

Section 24 — Inspect generated code

Numbers alone do not always explain why a model passed or failed.

Sometimes a model may fail because:

    the function signature was wrong
    the Rust syntax was invalid
    the logic changed
    the model added extra text
    the runtime error happened after compilation

That is why we also inspect the generated Rust code directly.

This section lets us look at:

    the generated Rust code
    the final Rust result
    the Python result
    runtime values
    any compiler or runtime errors

So this section helps us move from “what happened” to “why it happened.”

In [34]:
results_df[[
    "model_label",
    "task",
    "python_result",
    "rust_result",
    "python_execution_time_seconds",
    "rust_execution_time_seconds",
    "speedup_vs_python",
    "generated_rust",
    "stderr"
]]

,model_label,task,python_result,rust_result,python_execution_time_seconds,rust_execution_time_seconds,speedup_vs_python,generated_rust,stderr
0,Hugging Face - Qwen Coder 1.5B,count_primes,78498,78498,8.8909,0.271177,32.7863,fn count_primes_python(limit: usize) -> usize ...,
1,Google - Gemini 2.5 Flash,count_primes,78498,78498,5.7328,0.931014,6.1576,fn count_primes_python(limit: usize) -> usize ...,
2,Anthropic - Claude Sonnet 4.6,count_primes,None,None,NaN,NaN,NaN,,"Error code: 400 - {'type': 'error', 'error': {..."
3,OpenAI - GPT,count_primes,78498,78498,4.4185,0.274618,16.0896,fn count_primes_python(limit: usize) -> usize ...,


In [35]:
def interactive_rust_generator(model_key):
    try:
        task = BENCHMARK_TASK

        python_baseline = run_python_baseline(task)

        rust_code, generation_time = generate_rust(
            model_key=model_key,
            python_code=task["python_code"],
            rust_signature=task["rust_signature"],
            extra_instruction=task["extra_instruction"]
        )

        rust_result = compile_and_run_rust_benchmark(rust_code, task)

        compile_status = "Success" if rust_result["compile_success"] else "Failed"
        run_status = "Success" if rust_result["run_success"] else "Failed"

        if rust_result["rust_execution_time_seconds"] and rust_result["rust_execution_time_seconds"] > 0:
            speedup = python_baseline["python_execution_time_seconds"] / rust_result["rust_execution_time_seconds"]
            speedup_text = f"{speedup:.4f}x"
        else:
            speedup_text = "Not available"

        timing_info = (
            f"Generation time: {generation_time:.4f} sec\n"
            f"Compile time: {rust_result['compile_time_seconds']:.4f} sec\n"
            f"Python avg runtime: {python_baseline['python_execution_time_seconds']:.6f} sec\n"
            f"Rust runtime: {rust_result['rust_execution_time_seconds'] if rust_result['rust_execution_time_seconds'] is not None else 'Not available'}\n"
            f"Rust speedup vs Python: {speedup_text}"
        )

        final_output = (
            f"Expected output: {task['expected_output']}\n"
            f"Python result: {python_baseline['python_result']}\n"
            f"Rust result: {rust_result['rust_result']}"
        )

        error_output = rust_result["stderr"] if rust_result["stderr"] else "No errors"

        return rust_code, compile_status, run_status, final_output, timing_info, error_output

    except Exception as e:
        return "", "Failed", "Failed", "No output", "No timing available", str(e)

Section 26 — Define the Gradio app

In [36]:
demo = gr.Interface(
    fn=interactive_rust_generator,
    inputs=[
        gr.Dropdown(
            choices=list(MODEL_REGISTRY.keys()),
            value="hf_qwen_coder",
            label="Model key"
        )
    ],
    outputs=[
        gr.Code(label="Generated Rust code"),
        gr.Textbox(label="Compile status"),
        gr.Textbox(label="Run status"),
        gr.Textbox(label="Benchmark result summary", lines=6),
        gr.Textbox(label="Timing information", lines=8),
        gr.Textbox(label="Error output", lines=10)
    ],
    title="Python to Rust Benchmark Evaluation",
    description="Evaluate whether a model can convert a benchmark-style Python function into correct and faster Rust code."
)

Section 27 — Launch Gradio

In [37]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://925e07275c1c9b0864.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Running Python baseline...
Running Python baseline...
Running Python baseline...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://925e07275c1c9b0864.gradio.live
